# First L0 processor example, version==0.9.0

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-607

See the associated:

  * Python module: [first_l0_processor_with_optl.py](./first_l0_processor_with_optl.py)
  * YAML file: [first_l0_processor_with_optl.yaml](./first_l0_processor_with_optl.yaml)

## 1. Initialization

In [1]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

Prefect server URL used internally: http://prefect-server:4200/api
Prefect dashboard public URL: http://localhost:4200/dashboard


In [2]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *

init_demo()
init_dask_cluster_eopf(scale=4)

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import * 

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.1"


Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
Connecting to dask gateway for 'dask-eopf': http://dask-eopf:8000 ...
Create new dask cluster
Dask dashboard for 'dask-eopf': http://localhost:8702/clusters/134584a533ba4e1083ad1c5aec6b6bd1/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-eopf' are up: 0/4
Dask workers for 'dask-eopf' are up: 4/4


In [3]:
# Other imports
import getpass
import os
import os.path as osp
from resources import prefect_utils

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    PREFECT_BLOCK_S3.bucket_name,
    PREFECT_BLOCK_S3.bucket_folder,
    "users",
    os.environ.get("RSPY_HOST_USER", getpass.getuser()),
    "l0",
)
s3_config = osp.join(s3_base, "config")
s3_output = osp.join(s3_base, "output")

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./l0/config", s3_config)

# For each data: 
# input_config_dir: s3 bucket folder that contains the configuration files (NOT THE VOLUMINOUS DATA !).
# It will be downloaded locally.
# payload_file: input yaml configuration file to pass to the triggering. Local to the 'input_config_dir'.
# output_data_dir: s3 bucket directory that will contain the generated data.
s1_short = {
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.short.yaml",
    "output_data_dir": f"{s3_output}/s1.short",
}
s1 = {
    "input_config_dir": s3_config,
    "payload_file": "s1/iw_joborder.yaml",
    "output_data_dir": f"{s3_output}/s1",
}
s3 = {
    "input_config_dir": s3_config,
    "payload_file": "s3/s3_dordop_payload.yaml",
    "output_data_dir": f"{s3_output}/s3",
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

07:42:14.017 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/logging_config.yaml' to the bucket 'prefect-share' path 'sub/dir/users/ecombelles/l0/config/logging_config.yaml'.

07:42:14.022 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/l0_processor_configuration.yaml' to the bucket 'prefect-share' path 'sub/dir/users/ecombelles/l0/config/s3/l0_processor_configuration.yaml'.

07:42:14.024 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s3/s3_dordop_payload.yaml' to the bucket 'prefect-share' path 'sub/dir/users/ecombelles/l0/config/s3/s3_dordop_payload.yaml'.

07:42:14.026 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_joborder.yaml' to the bucket 'prefect-share' path 'sub/dir/users/ecombelles/l0/config/s1/iw_joborder.yaml'.

07:42:14.028 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_configuration.yaml' to the bucket 'prefect-share' path 'sub/dir/users/ecombelles/l0/config/s1/iw_configuration.yaml'.

07:42:14.031 | INFO    | prefect.S3Bucket - Uploading from 'l0/config/s1/iw_joborder.short.yaml' to the bucket 'prefect-share' path 'sub/dir/users/ecombelles/l0/config/s1/iw_joborder.short.yaml'.

07:42:14.109 | INFO    | prefect.S3Bucket - Uploaded 6 files from 'l0/config' to the bucket 'prefect-share' path 'sub/dir/users/ecombelles/l0/config/s1/iw_joborder.short.yaml'

In [4]:
# We use only the EOPF dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf
if local_mode:
    os.environ["DASK_GATEWAY_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_ADDRESS"]

# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

# Setup adaptive scaling
#dask_gateway.adapt_cluster(dask_cluster.name, minimum=1, maximum=scale)

## 2. Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [7]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{os.environ.get('RSPY_HOST_USER', getpass.getuser())}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{PREFECT_BLOCK_S3.bucket_name}/{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_BLOCK_S3.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}"

S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234
Upload local source code to: 's3://prefect-share/sub/dir/users/ecombelles/code'


In [8]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./first_l0_processor_with_optl.yaml"

07:46:04.952 | ERROR   | opentelemetry.instrumentation.instrumentor - DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.1"
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.4  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
07:46:05.990 | WARNING | prefect.utilities.templating - Value for placeholder 'RSPY_WEBSITE' not found in provided values. Please ensure that the placeholder is spelled correctly and that the corresponding value is provided.
07:46:05.991 | WARNING | prefect.utilities.templating - Value for placeholder 'RSPY_UAC_CHECK_URL' not found in provided values. Please ensure that the placeholder is

╭──────────────────────────────────────────────────────────────────────────────╮
│ Deployment                                                                   │
│ 'first-l0-processor-with-optl/sprint22-first-l0-processor-with-optl'         │
│ successfully created with id '21253c70-6c13-4840-a9eb-51a301694e5a'.         │
╰──────────────────────────────────────────────────────────────────────────────╯

View Deployment in UI: http://prefect-server:4200/deployments/deployment/21253c70-6c13-4840-a9eb-51a301694e5a


To schedule a run for this deployment, use the following command:

        $ prefect deployment run 
'first-l0-processor-with-optl/sprint22-first-l0-processor-with-optl'



In [9]:
deploy_name = "first-l0-processor-with-optl/sprint22-first-l0-processor-with-optl"
await prefect_utils.wait_for_deployment(deploy_name)

Finished deploying prefect flow: 'first-l0-processor-with-optl/sprint22-first-l0-processor-with-optl'


## 3. Run Prefect flow for S1 short data (~1 minute)

In [10]:
from importlib import reload
import first_l0_processor_with_optl
reload(first_l0_processor_with_optl)

output_data_dir = s1_short["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s1_short)

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.4  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Remove existing zarr products from: 's3://prefect-share/sub/dir/users/ecombelles/l0/output/s1.short'


In [11]:
%%bash -s "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

Creating flow run for deployment 
'first-l0-processor-with-optl/sprint22-first-l0-processor-with-optl'...
Created flow run 'practical-tuatara'.
└── UUID: 5e37efc3-9d42-4c6b-b7bb-e28793398371
└── Parameters: {'input_config_dir': 's3://prefect-share/sub/dir/users/ecombelles/l0/config', 'payload_file': 's1/iw_joborder.short.yaml', 'output_data_dir': 's3://prefect-share/sub/dir/users/ecombelles/l0/output/s1.short'}
└── Job Variables: {}
└── Scheduled start time: 2025-03-31 07:46:15 UTC (now)
└── URL: http://prefect-server:4200/runs/flow-run/5e37efc3-9d42-4c6b-b7bb-e28793398371
Watching flow run 'practical-tuatara'...


07:46:22.456 | INFO    | prefect - Flow run is in state 'Pending'
07:46:36.046 | INFO    | prefect - Flow run is in state 'Running'
07:47:54.910 | INFO    | prefect - Flow run is in state 'Completed'


Flow run finished successfully in 'Completed'.


In [12]:
print(f"Output products generated on: {output_data_dir!r}")

local_report_dir = osp.join("./l0", "reports", "s1.short")
print(f"Download reports locally: {local_report_dir!r}")
await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

Output products generated on: 's3://prefect-share/sub/dir/users/ecombelles/l0/output/s1.short'
Download reports locally: './l0/reports/s1.short'


## 4. Run Prefect flow for S1 full data (~30 minutes)

In [ ]:
output_data_dir = s1["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s1)

In [ ]:
%%bash -s "$from_cicd" "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line. Not from the ci/cd (too long).
if [[ "$1" == "False" ]]; then
    prefect deployment run "$2" --params "$3" --watch
fi

In [ ]:
if not from_cicd:
    print(f"Output products generated on: {output_data_dir!r}")
    
    local_report_dir = osp.join("./l0", "reports", "s1")
    print(f"Download reports locally: {local_report_dir!r}")
    await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 5. Run Prefect flow for S3 full data (~20 minutes)

In [ ]:
output_data_dir = s3["output_data_dir"]
print(f"Remove existing zarr products from: {output_data_dir!r}")
s3_delete(output_data_dir)

# Convert to json to trigger prefect flow
params_str = to_json(s3)

In [ ]:
%%bash -s "$from_cicd" "$deploy_name" "$params_str"
# Trigger a run for this flow from the command line. Not from the ci/cd (too long).
if [[ "$1" == "False" ]]; then
    prefect deployment run "$2" --params "$3" --watch
fi

In [ ]:
if not from_cicd:
    print(f"Output products generated on: {output_data_dir!r}")

    local_report_dir = osp.join("./l0", "reports", "s3")
    print(f"Download reports locally: {local_report_dir!r}")
    await s3_download_dir(osp.join(output_data_dir, "reports"), local_report_dir)

## 6. Shutdown the dask clusters

In [ ]:
# You can scale the clusters to 0 workers
dask_gateway.scale_cluster(dask_cluster.name, 0)

# Or shutdown the clusters
shutdown_dask_clusters(dask_gateway, dask_cluster.name)

# Close the python objects
close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.

## For testing only: reset the cluster and run the flow locally from Python

In [5]:
from importlib import reload
debug_flow = True

In [12]:
if debug_flow:
    shutdown_dask_clusters(dask_gateway, None)
    init_dask_cluster_eopf(scale=4)
    from resources.dask_utils import *
    dask_gateway = dask_gateway_eopf
    dask_client = dask_client_eopf
    dask_cluster = dask_cluster_eopf
    os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

In [6]:
if debug_flow:
    import first_l0_processor_with_optl
    reload(first_l0_processor_with_optl)
    results = first_l0_processor_with_optl.first_l0_processor_with_optl(**s1_short)
    display(results)

/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.4  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| numpy   | 2.2.4  | 1.26.4    | 1.26.4  |
| tornado | 6.3.3  | 6.4.2     | 6.4.2   |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


07:42:41.716 | INFO    | Flow run 'glittering-bloodhound' - Beginning flow run 'glittering-bloodhound' for flow 'first-l0-processor-with-optl'

07:42:41.723 | INFO    | Flow run 'glittering-bloodhound' - View at http://prefect-server:4200/runs/flow-run/4b388a80-d53d-4e52-a768-a97344b3367a

07:42:41.726 | WARNING | opentelemetry.trace - Overriding of current TracerProvider is not allowed

07:42:41.737 | ERROR   | opentelemetry.instrumentation.instrumentor - DependencyConflict: requested: "starlette >= 0.13, <0.15" but found: "starlette 0.46.1"

...

07:43:49.604 | INFO    | Task run 'dummy_catalog_save-e89' - Start catalog saving

07:43:50.622 | INFO    | Task run 'dummy_catalog_save-e89' - End (dummy) catalog saving:

07:43:50.628 | INFO    | Task run 'dummy_catalog_save-e89' - Finished in state Completed()

07:43:50.687 | INFO    | Flow run 'glittering-bloodhound' - Finished in state Completed()

{}